# AMML Ranking Competition 2026

Решение с Qwen3-Reranker-4B, контекстным BGE и BGE с окнами вокруг слов запроса. Подключите данные соревнования, выберите **GPU T4 x2** и включите Internet для загрузки весов. Запустите **Save & Run All**. Ноутбук обучает модели на данных соревнования и записывает `/kaggle/working/submission.csv`.

In [ ]:
import html
import json
import math
import os
import platform
import re
import unicodedata
from collections import Counter
from dataclasses import dataclass
from functools import lru_cache
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from nltk.stem.snowball import RussianStemmer


COMPETITION = "amml-ranking-competition-2026"
INPUT_FILES = ("train.csv", "test.csv", "sample_submission.csv")
NUMERIC_COLUMNS = ("passage_index", "article_passage_count", "article_char_count")
TEXT_FIELDS = ("query", "title", "text")
DOCUMENT_FIELDS = ("title", "text")
PAIR_COLUMNS = (
    "id", "query_id", "document_id", "article_id", "ingest_source",
    *TEXT_FIELDS, *NUMERIC_COLUMNS,
)
SEED = 2026
CPU_THREADS = min(os.cpu_count() or 1, 4)
WORD = re.compile(r"[^\W_]+", flags=re.UNICODE)
NUMBER = re.compile(r"\d+")
YEAR = re.compile(r"(?<!\d)(?:1[0-9]{3}|20[0-9]{2})(?!\d)")
TAG = re.compile(r"</?[a-z][a-z0-9:-]*(?:\s[^<>]*)?\s*/?>", re.IGNORECASE)
SPACES = re.compile(r"\s+")
MISSING = frozenset(("", "nan", "null", "[null]", "<null>", "none", "n/a", "<na>", "—"))
QUESTION_WORDS = ("кто", "что", "когда", "где", "куда", "откуда", "сколько", "почему", "как")
STEMMER = RussianStemmer()


def locate_input():
    override = os.environ.get("AMML_DATA_DIR")
    if override:
        candidates = [Path(override)]
    else:
        candidates = sorted({path.parent for path in Path("/kaggle/input").rglob(INPUT_FILES[-1])})
    matches = [folder for folder in candidates if all((folder / name).is_file() for name in INPUT_FILES)]
    competition_matches = [folder for folder in matches if COMPETITION in folder.parts]
    matches = competition_matches or matches
    if len(matches) != 1:
        raise FileNotFoundError("Attach the competition data or set AMML_DATA_DIR to its folder.")
    return matches[0]


def read_pairs(path, *, labeled):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    required = {*PAIR_COLUMNS, *(["relevance"] if labeled else [])}
    if missing := required.difference(frame.columns):
        raise ValueError(f"{path.name}: missing columns {sorted(missing)}")
    if frame["id"].eq("").any() or frame["id"].duplicated().any() or frame["query_id"].eq("").any():
        raise ValueError(f"{path.name}: expected unique pair IDs and a query ID for every row")
    if labeled:
        label = pd.to_numeric(frame["relevance"], errors="coerce")
        if not label.isin([0, 1]).all():
            raise ValueError("Expected binary relevance labels.")
        frame["relevance"] = label.astype(np.int8)
    return frame.sort_values(["query_id", "id"], kind="stable").reset_index(drop=True)


def load_data():
    folder = locate_input()
    train = read_pairs(folder / INPUT_FILES[0], labeled=True)
    test = read_pairs(folder / INPUT_FILES[1], labeled=False)
    sample = pd.read_csv(folder / INPUT_FILES[2], dtype={"id": str}, keep_default_na=False)
    return train, test, sample


def clean_text(value):
    value = unicodedata.normalize("NFKC", html.unescape(value)).translate(
        str.maketrans({"\u200b": "", "\ufeff": "", "\u00ad": ""})
    )
    value = SPACES.sub(" ", TAG.sub(" ", value)).strip()
    return "" if value.casefold() in MISSING else value


def clean_number(column):
    value = column.map(clean_text).str.replace(r"(?<=\d)\s+(?=\d)", "", regex=True)
    thousands = value.str.fullmatch(r"\d{1,3}(?:,\d{3})+")
    value = value.where(~thousands, value.str.replace(",", "", regex=False))
    value = value.str.replace(",", ".", regex=False)
    number = pd.to_numeric(value, errors="coerce").astype(float)
    return number.where(np.isfinite(number) & number.ge(0) & number.mod(1).eq(0))


def prepare_pairs(raw):
    frame = raw.copy()
    for field in TEXT_FIELDS:
        frame[field] = raw[field].map(clean_text)
        frame[f"{field}_missing"] = frame[field].eq("")
    known_queries = frame.loc[~frame["query_missing"], ["query_id", "query"]].copy()
    known_queries["normalized"] = known_queries["query"].str.casefold().str.replace("ё", "е")
    known = known_queries.groupby("query_id", sort=False).agg(
        query=("query", "first"), variants=("normalized", "nunique")
    )
    recovered = frame["query_id"].map(known.loc[known["variants"].eq(1), "query"]).fillna("")
    frame["query"] = frame["query"].where(~frame["query_missing"], recovered)
    for field in NUMERIC_COLUMNS:
        frame[field] = clean_number(raw[field])
    frame["ingest_source"] = raw["ingest_source"].map(clean_text).str.casefold()
    return frame


@lru_cache(maxsize=180_000)
def stem(word):
    return STEMMER.stem(word)


@dataclass
class Terms:
    words: tuple
    stems: tuple
    counts: Counter
    numbers: frozenset
    text: str


@lru_cache(maxsize=94_000)
def analyze_text(text):
    normalized = text.casefold().replace("ё", "е")
    words = tuple(WORD.findall(normalized))
    stems = tuple(stem(word) for word in words)
    return Terms(words, stems, Counter(stems), frozenset(NUMBER.findall(normalized)), normalized)


@lru_cache(maxsize=10_000)
def char_trigrams(text):
    compact = " ".join(WORD.findall(text.casefold().replace("ё", "е")))
    return frozenset(compact[i:i + 3] for i in range(len(compact) - 2))


@dataclass
class CollectionStats:
    idf: dict
    unseen_idf: float
    avg_length: float


def collection_stats(texts):
    frequency = Counter()
    document_count = total_tokens = 0
    for text in dict.fromkeys(texts):
        tokens = analyze_text(text)
        if tokens.stems:
            frequency.update(tokens.counts.keys())
            document_count += 1
            total_tokens += len(tokens.stems)
    idf = {
        token: math.log1p((document_count - count + 0.5) / (count + 0.5))
        for token, count in frequency.items()
    }
    return CollectionStats(idf, math.log1p((document_count + 0.5) / 0.5),
                           max(1.0, total_tokens / max(document_count, 1)))


PAIR_FEATURES = (
    "bm25", "idf_coverage", "word_coverage", "stem_coverage", "lead_coverage",
    "stem_bigram_coverage", "char_trigram_coverage", "query_phrase", "document_terms",
    "document_chars", "shared_numbers", "document_numbers", "has_year",
)


def pair_features(query_text, document_text, stats):
    query, doc = analyze_text(query_text), analyze_text(document_text)
    matched = query.counts.keys() & doc.counts.keys()
    idf = {term: stats.idf.get(term, stats.unseen_idf) for term in query.counts}
    norm = 1.2 * (0.25 + 0.75 * len(doc.stems) / stats.avg_length)
    bm25 = sum(idf[term] * 2.2 * doc.counts[term] / (norm + doc.counts[term])
               for term in query.counts if term in matched)
    word_overlap = len(set(query.words).intersection(doc.words))
    query_bigrams = set(zip(query.stems, query.stems[1:]))
    bigram_hits = sum(bigram in query_bigrams for bigram in set(zip(doc.stems, doc.stems[1:])))
    grams = char_trigrams(query_text)
    return (
        bm25,
        sum(idf[term] for term in query.counts if term in matched) / (sum(idf.values()) or 1),
        word_overlap / max(len(set(query.words)), 1),
        len(matched) / max(len(query.counts), 1),
        len(query.counts.keys() & set(doc.stems[:45])) / max(len(query.counts), 1),
        bigram_hits / max(len(query_bigrams), 1),
        sum(gram in doc.text for gram in grams) / max(len(grams), 1),
        float(bool(query.text) and (query.text.rstrip("?!.") + " ") in (doc.text + " ")),
        len(doc.stems), len(document_text), len(query.numbers & doc.numbers),
        len(doc.numbers), float(bool(YEAR.search(doc.text))),
    )


def fit_feature_state(training):
    return {field: collection_stats(training[field]) for field in DOCUMENT_FIELDS}


def make_features(frame, state, *, include_metadata=True):
    lexical = []
    for field in DOCUMENT_FIELDS:
        values = [pair_features(query, text, state[field]) for query, text in zip(frame["query"], frame[field])]
        lexical.append(pd.DataFrame(values, index=frame.index,
                                   columns=[f"{field}_{name}" for name in PAIR_FEATURES], dtype=np.float32))
    features = pd.concat(lexical, axis=1)
    query_terms = frame["query"].map(lambda text: analyze_text(text).words)
    features["query_terms"] = query_terms.map(len)
    features["query_chars"] = frame["query"].str.len()
    features["question_word"] = query_terms.map(
        lambda words: words[0] if words and words[0] in QUESTION_WORDS else "other"
    )
    for field in TEXT_FIELDS:
        features[f"{field}_missing"] = frame[f"{field}_missing"].astype(np.int8)

    comparable = [f"{field}_{signal}" for field in DOCUMENT_FIELDS
                  for signal in ("bm25", "idf_coverage", "char_trigram_coverage")]
    by_query = features[comparable].groupby(frame["query_id"], sort=False)
    features = pd.concat([
        features,
        by_query.rank(ascending=False, method="average", pct=True).add_suffix("_query_pct"),
        (features[comparable] - by_query.transform("max")).add_suffix("_from_best"),
    ], axis=1)
    if include_metadata:
        for field in NUMERIC_COLUMNS:
            features[field] = frame[field].astype(np.float32)
        features["relative_position"] = frame["passage_index"] / frame["article_passage_count"].where(
            frame["article_passage_count"].gt(0)
        )
        features["candidates_in_article"] = frame.groupby(["query_id", "article_id"], sort=False)["id"].transform(
            "size"
        ).where(frame["article_id"].ne(""), 1)
        features["ingest_source"] = frame["ingest_source"]
    return features


def validate_template(test, sample):
    if list(sample.columns) != ["id", "prediction"] or sample["id"].duplicated().any():
        raise ValueError("Expected sample_submission.csv with unique id and prediction columns.")
    if len(sample) != len(test) or set(sample["id"]) != set(test["id"]):
        raise ValueError("Test IDs do not match the submission template.")


def write_submission(test, sample, prediction, path):
    validate_template(test, sample)
    scores = np.asarray(prediction, dtype=np.float64)
    if scores.shape != (len(test),) or not np.isfinite(scores).all():
        raise ValueError("Expected one finite score for every test pair.")
    scores_by_id = pd.Series(scores, index=test["id"])
    submission = sample[["id"]].assign(prediction=sample["id"].map(scores_by_id))
    submission.to_csv(path, index=False, float_format="%.17g")
    print(f"Wrote {len(submission):,} predictions to {path}", flush=True)

In [ ]:
import gc
import os
import re
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import torch
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer


QWEN = "Qwen/Qwen3-Reranker-4B"
QWEN_REV = "22e6836"
BGE = "BAAI/bge-reranker-v2-m3"
BGE_REV = "953dc6f"
MODEL_CACHE = Path(os.environ.get("AMML_MODEL_CACHE", "/kaggle/temp/amml_models"))
INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"
PREFIX = '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"


def configure_accelerator():
    if torch.cuda.device_count() < 2:
        raise RuntimeError("The notebook requires Kaggle GPU T4 x2.")
    torch.set_num_threads(CPU_THREADS)
    print(f"CUDA devices: {[torch.cuda.get_device_name(i) for i in range(2)]}", flush=True)
    print({name: version(name) for name in ("torch", "transformers", "tokenizers", "huggingface-hub", "safetensors")}, flush=True)


def document_inputs(rows, snippets=None):
    text = rows["text"].tolist() if snippets is None else snippets
    return [f"{title}\n{passage}".strip() for title, passage in zip(rows["title"], text)]


def select_answer_window(query, text):
    words = list(re.finditer(r"\S+", text))
    if len(words) <= 110:
        return text
    query_terms = set(analyze_text(query).stems)
    start_positions = list(range(0, len(words) - 90, 45))
    start_positions.append(len(words) - 90)
    windows = [text[words[start].start():words[start + 89].end()] for start in start_positions]
    return max(windows, key=lambda window: len(analyze_text(window).counts.keys() & query_terms))


def qwen_input(tokenizer, rows):
    prefix = tokenizer.encode(PREFIX, add_special_tokens=False)
    suffix = tokenizer.encode(SUFFIX, add_special_tokens=False)
    inputs = [f"<Instruct>: {INSTRUCTION}\n<Query>: {query}\n<Document>: {doc}"
              for query, doc in zip(rows["query"], document_inputs(rows))]
    tokens = tokenizer(inputs, add_special_tokens=False, return_attention_mask=False,
                       truncation=True, max_length=416 - len(prefix) - len(suffix))["input_ids"]
    return [{"input_ids": prefix + ids + suffix} for ids in tokens]


def bert_input(tokenizer, rows, snippets=None):
    encoded = tokenizer(rows["query"].tolist(), document_inputs(rows, snippets),
                        truncation=True, max_length=384, add_special_tokens=True)
    return [dict(zip(encoded.keys(), fields)) for fields in zip(*encoded.values())]


def load_rank_encoder(repo, revision, device, *, causal):
    print(f"LOAD {repo}@{revision} on {device}", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(repo, revision=revision, cache_dir=MODEL_CACHE,
                                              trust_remote_code=False)
    constructor = AutoModelForCausalLM if causal else AutoModelForSequenceClassification
    model = constructor.from_pretrained(repo, revision=revision, cache_dir=MODEL_CACHE, use_safetensors=True,
                                        trust_remote_code=False, dtype=torch.float16,
                                        device_map={"": device}, attn_implementation="sdpa").eval()
    if causal:
        tokenizer.padding_side = "left"
    print(f"READY {repo} on {device}", flush=True)
    return tokenizer, model


def score_batches(tokenizer, model, encoded, device, name, *, causal=False, batch_size=48):
    @torch.inference_mode()
    def score_batch(numbers):
        padded = tokenizer.pad([encoded[index] for index in numbers], padding=True,
                               pad_to_multiple_of=8, return_tensors="pt").to(device)
        if causal:
            logits = model(**padded, use_cache=False, logits_to_keep=1).logits[:, -1].float()
            yes = tokenizer.convert_tokens_to_ids("yes")
            no = tokenizer.convert_tokens_to_ids("no")
            return (logits[:, yes] - logits[:, no]).cpu().numpy()
        return model(**padded).logits.reshape(-1).float().cpu().numpy()

    order = sorted(range(len(encoded)), key=lambda index: len(encoded[index]["input_ids"]))
    scores = np.empty(len(encoded), dtype=np.float32)
    start = perf_counter()
    position = 0
    last_report = start
    while position < len(order):
        indexes = order[position:position + batch_size]
        memory_error = False
        try:
            scores[indexes] = score_batch(indexes)
        except torch.cuda.OutOfMemoryError:
            memory_error = True
        if memory_error:
            if batch_size <= 1:
                raise RuntimeError(f"{name} failed with batch_size=1 on {device}")
            batch_size //= 2
            gc.collect()
            torch.cuda.empty_cache()
            print(f"{name} retry with batch size {batch_size}", flush=True)
            continue
        position += len(indexes)
        now = perf_counter()
        if position == len(order) or (position >= 384 and (last_report == start or now - last_report >= 90)):
            rate = position / (now - start)
            print(f"ENCODE {name}: {position:,}/{len(order):,}, {rate:.1f} pairs/s, est. {(len(order)-position)/rate/60:.1f} min left", flush=True)
            last_report = now
    if not np.isfinite(scores).all():
        raise RuntimeError(f"{name} produced non-finite scores")
    return scores


def score_qwen(rows, indices, gpu):
    device = torch.device(f"cuda:{gpu}")
    tokenizer, model = load_rank_encoder(QWEN, QWEN_REV, device, causal=True)
    encoded = qwen_input(tokenizer, rows.iloc[indices])
    scores = score_batches(tokenizer, model, encoded, device, f"Qwen4B/{gpu}", causal=True, batch_size=32)
    return scores


def score_bge(rows, gpu):
    device = torch.device(f"cuda:{gpu}")
    tokenizer, model = load_rank_encoder(BGE, BGE_REV, device, causal=False)
    full = score_batches(tokenizer, model, bert_input(tokenizer, rows), device, "BGE/full", batch_size=64)
    span = full.copy()
    long_rows = rows.index[rows["text"].str.len() > 1000].to_numpy()
    if len(long_rows):
        subset = rows.loc[long_rows]
        windows = [select_answer_window(q, text) for q, text in zip(subset["query"], subset["text"])]
        span[long_rows] = score_batches(tokenizer, model, bert_input(tokenizer, subset, windows),
                                       device, "BGE/answer-window", batch_size=64)
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return full, span


def encode_semantics(rows, cache_path):
    if cache_path.is_file():
        cached = pd.read_parquet(cache_path)
        if cached["id"].tolist() != rows["id"].tolist():
            raise ValueError("Cached scores belong to a different set of pairs")
        return cached.drop(columns="id")

    order = np.random.default_rng(SEED).permutation(len(rows))
    cut = int(0.59 * len(order))
    left, right = order[:cut], order[cut:]

    def second_gpu():
        bge, span = score_bge(rows, 1)
        return bge, span, score_qwen(rows, right, 1)

    print(f"SCORING {len(rows):,} pairs on two GPUs", flush=True)
    with ThreadPoolExecutor(max_workers=2) as executor:
        first = executor.submit(score_qwen, rows, left, 0)
        second = executor.submit(second_gpu)
        qwen = np.empty(len(rows), dtype=np.float32)
        qwen[left] = first.result()
        bge, span, qwen[right] = second.result()
    result = pd.DataFrame({"qwen": qwen, "bge": bge, "bge_window": span})
    result.assign(id=rows["id"]).to_parquet(cache_path, index=False)
    print(f"SEMANTICS SAVED {cache_path}", flush=True)
    return result

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from threading import Lock
from time import perf_counter
import gc
import os

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import torch
from torch import nn
from torch.nn import functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer


MODEL_CACHE = Path(os.environ.get("AMML_MODEL_CACHE", "/kaggle/temp/amml_models"))
INITIALIZATION_LOCK = Lock()
SEMANTIC_COLUMNS = ("qwen", "bge", "bge_window")


@dataclass(frozen=True)
class EncoderConfig:
    repo: str
    revision: str
    max_length: int
    layers: int
    learning_rate: float
    max_pairs: int
    max_tokens: int


ENCODERS = {
    "joint_bge": EncoderConfig("BAAI/bge-reranker-v2-m3", "953dc6f", 384, 6, 1.25e-5, 44, 14848),
}


def standardize_scores(scores, groups):
    values = pd.Series(np.asarray(scores, dtype=np.float64), index=groups.index)
    stats = values.groupby(groups, sort=False)
    return ((values - stats.transform("mean")) / stats.transform("std").replace(0, np.nan)).fillna(0)


def candidate_groups(rows):
    return [positions.to_numpy() for positions in rows.groupby("query_id", sort=False).groups.values()]


class CandidateData:
    def __init__(self, rows, semantics):
        self.rows = rows
        self.labels = rows["relevance"].fillna(0).to_numpy(dtype=np.float32)
        article = "article:" + rows["article_id"]
        article = article.where(rows["article_id"].ne(""), "missing:" + rows["id"])
        self.article = pd.factorize(article, sort=False)[0]
        self.position = rows["passage_index"].fillna(-1).to_numpy(dtype=np.float32)
        groups = rows["query_id"]
        semantic = np.column_stack([standardize_scores(semantics[name], groups) for name in SEMANTIC_COLUMNS])
        self.prior = (semantic @ np.array([0.55, 0.30, 0.15])).astype(np.float32)
        passages = rows.groupby([groups, article], sort=False)["id"].transform("size")
        candidates = rows.groupby(groups, sort=False)["id"].transform("size")
        title_size = rows["title"].str.len()
        text_size = rows["text"].str.len()
        position = rows["passage_index"] / rows["article_passage_count"].where(rows["article_passage_count"].gt(0))
        self.side = np.column_stack([
            semantic, semantic[:, 0] - semantic[:, 1], semantic[:, 2] - semantic[:, 1],
            passages / candidates, position.clip(0, 2).fillna(-1),
            standardize_scores(np.log1p(text_size), groups), standardize_scores(np.log1p(title_size), groups),
            np.log1p(rows["query"].str.len()) / 5.0, rows["title_missing"],
        ]).astype(np.float32)


def tokenize_pairs(data, tokenizer, config):
    rows = data.rows
    documents = [f"{title}\n{text}".strip() for title, text in zip(rows["title"], rows["text"])]
    queries = rows["query"].tolist()
    tokens = tokenizer(queries, documents, truncation=True, max_length=config.max_length)
    return [dict(zip(tokens, pair)) for pair in zip(*tokens.values())]


def batch_groups(groups, lengths, max_pairs, max_tokens, rng=None):
    widths = [int(lengths[group].max()) for group in groups]
    order = list(range(len(groups))) if rng is None else rng.permutation(len(groups)).tolist()
    order.sort(key=lambda offset: widths[offset] if rng is None else widths[offset] // 48)
    batches, batch, count, longest = [], [], 0, 0
    for offset in order:
        group, length = groups[offset], widths[offset]
        proposed_width = max(length, longest)
        proposed_count = count + len(group)
        if batch and (proposed_count > max_pairs or proposed_count * proposed_width > max_tokens):
            batches.append(batch)
            batch, count, longest = [], 0, 0
        batch.append(group)
        count += len(group)
        longest = max(longest, length)
    if batch:
        batches.append(batch)
    if rng is not None:
        rng.shuffle(batches)
    return batches


def pack_candidates(tokenizer, encoded, data, groups, device, *, context):
    indices = np.concatenate(groups)
    width = max(map(len, groups))
    lookup = np.full((len(groups), width), -1, dtype=np.int64)
    offset = 0
    for row, group in zip(lookup, groups):
        row[:len(group)] = np.arange(offset, offset + len(group))
        offset += len(group)
    mask = lookup >= 0
    lookup = lookup.clip(min=0)
    positions = indices[lookup]
    tokens = tokenizer.pad([encoded[index] for index in indices], padding=True, pad_to_multiple_of=8,
                           return_tensors="pt").to(device)
    gather = torch.from_numpy(lookup).to(device)
    valid = torch.from_numpy(mask).to(device)
    targets = torch.from_numpy(data.labels[positions] * mask).to(device)
    side = prior = relations = None
    if context:
        articles = data.article[positions]
        offsets = data.position[positions]
        same = articles[:, :, None] == articles[:, None, :]
        usable = (offsets[:, :, None] >= 0) & (offsets[:, None, :] >= 0)
        distance = np.abs(offsets[:, :, None] - offsets[:, None, :])
        proximity = same * usable * np.exp(-distance / 4.0)
        relation_values = np.stack([same, proximity], axis=-1).astype(np.float32)
        side = torch.from_numpy(data.side[indices]).to(device)
        prior = torch.from_numpy(data.prior[positions]).to(device)
        relations = torch.from_numpy(relation_values).to(device)
    return indices, tokens, gather, valid, targets, side, prior, relations


class ArticleAttention(nn.Module):
    def __init__(self, width=128, heads=4):
        super().__init__()
        self.width, self.heads = width, heads
        self.norm = nn.LayerNorm(width)
        self.qkv = nn.Linear(width, width * 3)
        self.out = nn.Linear(width, width)
        self.relation_weight = nn.Parameter(torch.zeros(heads, 2))
        self.mlp = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 2 * width), nn.GELU(),
                                 nn.Dropout(0.10), nn.Linear(2 * width, width))

    def forward(self, inputs, valid, relations):
        batch, size, _ = inputs.shape
        qkv = self.qkv(self.norm(inputs)).reshape(batch, size, 3, self.heads, self.width // self.heads)
        queries, keys, values = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        bias = torch.einsum("bijd,hd->bhij", relations.to(queries.dtype), self.relation_weight.to(queries.dtype))
        bias = bias.masked_fill(~valid[:, None, None, :], -10_000.0)
        attended = F.scaled_dot_product_attention(queries, keys, values, attn_mask=bias,
                                                   dropout_p=0.08 if self.training else 0.0)
        residual = inputs + self.out(attended.transpose(1, 2).reshape(batch, size, self.width))
        return residual + self.mlp(residual)


class BgeQueryRanker(nn.Module):
    def __init__(self, pretrained, side_width):
        super().__init__()
        self.backbone = pretrained.base_model
        self.pair_head = pretrained.classifier
        self.project = nn.Sequential(nn.Linear(pretrained.config.hidden_size + side_width, 128),
                                     nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10))
        self.attention = ArticleAttention()
        self.score = nn.Sequential(nn.Linear(256, 96), nn.GELU(), nn.Dropout(0.1), nn.Linear(96, 1))
        nn.init.zeros_(self.score[-1].weight)
        nn.init.zeros_(self.score[-1].bias)

    def forward(self, tokens, gather, valid, side, prior, relations):
        hidden = self.backbone(**tokens).last_hidden_state
        raw = self.pair_head(hidden)[:, 0].float()
        features = self.project(torch.cat([hidden[:, 0], side.to(hidden.dtype)], dim=-1))[gather]
        features = self.attention(features, valid, relations)
        average = (features * valid.unsqueeze(-1)).sum(1, keepdim=True) / valid.sum(1)[:, None, None]
        correction = self.score(torch.cat([features, average.expand_as(features)], dim=-1)).squeeze(-1).float()
        return (raw[gather] + 0.55 * prior + correction).masked_fill(~valid, -10_000.0)


def make_model(name, device, side_width, seed):
    config = ENCODERS[name]
    pretrained = AutoModelForSequenceClassification.from_pretrained(
        config.repo, revision=config.revision, cache_dir=MODEL_CACHE, use_safetensors=True,
        trust_remote_code=False, dtype=torch.float32, device_map={"": device}, attn_implementation="sdpa",
    )
    pretrained.requires_grad_(False)
    layers = pretrained.base_model.encoder.layer
    with INITIALIZATION_LOCK, torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(seed)
        model = BgeQueryRanker(pretrained, side_width).to(device)
    frozen = [pretrained.base_model.embeddings, *layers[:-config.layers]]
    pretrained.classifier.requires_grad_(True)
    head = list(model.project.parameters()) + list(model.attention.parameters()) + list(model.score.parameters())
    for layer in layers[-config.layers:]:
        layer.requires_grad_(True)
    head_ids = {id(parameter) for parameter in head}
    encoder_params = [parameter for parameter in model.parameters() if parameter.requires_grad and id(parameter) not in head_ids]
    groups = [{"params": encoder_params, "lr": config.learning_rate}, {"params": head, "lr": 0.00018}]
    print(f"LOADED {name} on {device} | trainable {sum(p.numel() for p in encoder_params + head):,}", flush=True)
    return model, frozen, groups


def ranking_loss(logits, labels, valid):
    width = logits.shape[1]
    positions = torch.arange(width, device=logits.device)
    discount = 1 / torch.log2(positions + 2.0)
    discount[10:] = 0
    positives = labels.sum(1)
    idcg = torch.cat([discount.new_zeros(1), discount.cumsum(0)])[positives.long()]
    order = logits.detach().argsort(dim=1, descending=True, stable=True)
    ranks = torch.empty_like(order).scatter_(1, order, positions.expand_as(order))
    gains = discount[ranks]
    swap = (gains[:, :, None] - gains[:, None, :]).abs()
    pairs = (labels[:, :, None] > labels[:, None, :]) & valid[:, :, None] & valid[:, None, :]
    weights = pairs * (swap / idcg.clamp_min(1e-6)[:, None, None] + 0.02)
    margin = logits[:, None, :] - logits[:, :, None]
    pair_loss = (F.softplus(margin) * weights).sum((1, 2)) / pairs.sum((1, 2)).clamp_min(1)
    distribution = labels / positives.clamp_min(1)[:, None]
    list_loss = -(distribution * F.log_softmax(logits, dim=1)).sum(1)
    point_loss = F.binary_cross_entropy_with_logits(logits[valid], labels[valid])
    return (pair_loss + 0.11 * list_loss).mean() + 0.09 * point_loss


def forward_batch(model, batch):
    _, tokens, gather, valid, _, side, prior, relations = batch
    with torch.amp.autocast("cuda", dtype=torch.float16):
        return model(tokens, gather, valid, side, prior, relations)


@torch.inference_mode()
def evaluate_candidates(model, tokenizer, encoded, data, groups, lengths, device, config, context):
    model.eval()
    predictions = np.full(len(data.rows), np.nan, dtype=np.float32)
    batches = batch_groups(groups, lengths, min(72, config.max_pairs * 2), config.max_tokens * 2)
    for groups_batch in batches:
        batch = pack_candidates(tokenizer, encoded, data, groups_batch, device, context=context)
        prediction = forward_batch(model, batch)[batch[3]].cpu().numpy()
        predictions[batch[0]] = prediction
    return predictions


class TrainingBudgetExhausted(RuntimeError):
    pass


def train_encoder(name, tokenizer, encoded, lengths, data, fit, holdout, *, gpu, tag, epochs,
                  deadline, assess=False, batch_scale=1.0, seed_offset=0):
    device = torch.device(f"cuda:{gpu}")
    config = ENCODERS[name]
    context = name == "joint_bge"
    fit_groups, holdout_groups = candidate_groups(fit), candidate_groups(holdout)
    rng = np.random.default_rng(SEED + seed_offset)
    torch.cuda.manual_seed(SEED + seed_offset)
    model, frozen, parameters = make_model(name, device, data.side.shape[1], SEED + seed_offset)
    optimizer = torch.optim.AdamW(parameters, weight_decay=0.01)
    rates = [group["lr"] for group in optimizer.param_groups]
    trainable = [parameter for group in parameters for parameter in group["params"]]
    scaler = torch.amp.GradScaler("cuda")
    outputs = {}
    for epoch in range(1, epochs + 1):
        started = perf_counter()
        model.train()
        for layer in frozen:
            layer.eval()
        batches = batch_groups(fit_groups, lengths, max(12, int(config.max_pairs * batch_scale)),
                               int(config.max_tokens * batch_scale), rng)
        warmup = max(20, len(batches) // 16)
        for step, groups in enumerate(batches, 1):
            if step % 24 == 1 and perf_counter() > deadline:
                raise TrainingBudgetExhausted(f"{name} {tag}: reached the reserved GPU time")
            factor = min(1.0, step / warmup) if epoch == 1 else 0.85 ** (epoch - 1)
            for rate, group in zip(rates, optimizer.param_groups):
                group["lr"] = rate * factor
            batch = pack_candidates(tokenizer, encoded, data, groups, device, context=context)
            optimizer.zero_grad(set_to_none=True)
            logits = forward_batch(model, batch)
            loss = ranking_loss(logits, batch[4], batch[3])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer)
            scaler.update()
            if step == 1:
                print(f"FIRST STEP {name} {tag} e{epoch} | loss {float(loss.detach()):.5f}", flush=True)
            if step == 40 or step % 180 == 0:
                seconds = perf_counter() - started
                eta = (len(batches) - step) * seconds / step / 60
                print(f"TRAIN {name} {tag} e{epoch}: {step}/{len(batches)} | epoch left ~{eta:.1f}m", flush=True)
        if epoch == epochs:
            scaled = EncoderConfig(config.repo, config.revision, config.max_length, config.layers, config.learning_rate,
                                   int(config.max_pairs * batch_scale), int(config.max_tokens * batch_scale))
            prediction = evaluate_candidates(model, tokenizer, encoded, data, holdout_groups, lengths,
                                             device, scaled, context)
            if not np.isfinite(prediction[holdout.index]).all():
                raise ValueError(f"{name} {tag} produced invalid predictions")
            outputs[epoch] = prediction
            print(f"EPOCH {name} {tag} e{epoch} finished {(perf_counter()-started)/60:.1f}m", flush=True)
    return outputs


def run_encoder(name, data, jobs, gpu, deadline, *, seed_offset=0):
    config = ENCODERS[name]
    with torch.cuda.device(gpu):
        tokenizer = AutoTokenizer.from_pretrained(config.repo, revision=config.revision, cache_dir=MODEL_CACHE,
                                                  trust_remote_code=False)
        encoded = tokenize_pairs(data, tokenizer, config)
        lengths = np.array([len(tokens["input_ids"]) for tokens in encoded])
        print(f"READY {name} on {torch.cuda.get_device_name(gpu)} | tokenized {len(encoded):,}", flush=True)
        results = {}
        scale = 1.0
        for tag, fit, holdout, epochs, assess in jobs:
            print(f"START {name} {tag} | {len(fit):,} train, {len(holdout):,} evaluate", flush=True)
            for attempt in range(2):
                try:
                    outputs = train_encoder(name, tokenizer, encoded, lengths, data, fit, holdout, gpu=gpu,
                                            tag=tag, epochs=epochs, deadline=deadline, assess=assess,
                                            batch_scale=scale, seed_offset=seed_offset)
                except torch.cuda.OutOfMemoryError:
                    if attempt:
                        raise
                else:
                    break
                scale *= 0.65
                gc.collect()
                torch.cuda.empty_cache()
                print(f"RETRY {name} {tag} with {scale:.2f} batch budget", flush=True)
            results[tag] = outputs
            gc.collect()
            torch.cuda.empty_cache()
        return results

In [ ]:
SEMANTIC_COLUMNS = ("qwen", "bge", "bge_window")


def standardized(scores, groups):
    series = pd.Series(np.asarray(scores), index=groups.index)
    by_query = series.groupby(groups, sort=False)
    spread = by_query.transform("std").replace(0, np.nan)
    return ((series - by_query.transform("mean")) / spread).fillna(0)


def semantic_prior(rows, semantic):
    groups = rows["query_id"]
    values = {column: standardized(semantic[column], groups) for column in SEMANTIC_COLUMNS}
    return 0.55 * values["qwen"] + 0.30 * values["bge"] + 0.15 * values["bge_window"]


def paired_neighbors(rows, article_key, values):
    outputs = np.full((len(rows), values.shape[1] * 2), np.nan, dtype=np.float32)
    article = article_key.to_numpy()
    position = rows["passage_index"].to_numpy(dtype=float)
    for group_positions in rows.groupby("query_id", sort=False).indices.values():
        if len(group_positions) < 2:
            continue
        local_position = position[group_positions]
        local_scores = values[group_positions]
        local_article = article[group_positions]
        same_article = local_article[:, None] == local_article[None, :]
        np.fill_diagonal(same_article, False)
        distances = np.abs(local_position[:, None] - local_position[None, :])
        weights = np.where(same_article & np.isfinite(distances), np.exp(-np.nan_to_num(distances) / 4), 0)
        totals = weights.sum(axis=1, keepdims=True)
        avg = np.divide(weights @ local_scores, totals, out=np.full_like(local_scores, np.nan), where=totals > 0)
        best = np.where(same_article[:, :, None], local_scores[None, :, :], -np.inf).max(axis=1)
        best[~np.isfinite(best)] = np.nan
        outputs[group_positions] = np.column_stack([avg, best])
    return outputs


def context_features(rows, base, semantic):
    scores = semantic.loc[rows.index, list(SEMANTIC_COLUMNS)]
    article_key = "article:" + rows["article_id"].astype(str)
    article_key = article_key.where(rows["article_id"].ne(""), "pair:" + rows["id"])
    groups = rows["query_id"]
    block = {}
    title_key = rows["title"].str.casefold().str.replace("ё", "е")
    same_title = rows.groupby([groups, title_key], sort=False)["id"].transform("size")
    block["same_title_in_query"] = same_title.where(title_key.ne(""), 1)
    block["articles_in_query"] = article_key.groupby(groups, sort=False).transform("nunique")
    block["candidates_in_query"] = article_key.groupby(groups, sort=False).transform("size")
    article_sizes = article_key.groupby([groups, article_key], sort=False).transform("size")
    block["article_size_share"] = article_sizes / block["candidates_in_query"]
    block["largest_article_size"] = article_sizes.groupby(groups, sort=False).transform("max")
    block["bge_window_improvement"] = scores["bge_window"] - scores["bge"]
    block["semantic_consensus"] = semantic_prior(rows, scores)

    for name in (*SEMANTIC_COLUMNS, "text_bm25", "title_bm25"):
        value = scores[name] if name in scores else base[name]
        by_query = value.groupby(groups, sort=False)
        by_article = value.groupby([groups, article_key], sort=False)
        article_max = by_article.transform("max")
        article_first = article_max.groupby(groups, sort=False).transform("max")
        block[f"{name}_query_z"] = standardized(value, groups)
        block[f"{name}_query_rank"] = by_query.rank(ascending=False, method="average", pct=True)
        block[f"{name}_query_max"] = by_query.transform("max")
        block[f"{name}_query_std"] = by_query.transform("std")
        block[f"{name}_to_query_best"] = value - by_query.transform("max")
        block[f"{name}_article_max"] = article_max
        block[f"{name}_article_mean"] = by_article.transform("mean")
        block[f"{name}_to_article_best"] = value - article_max
        block[f"{name}_article_to_query_best"] = article_max - article_first
        block[f"{name}_peer_mean"] = (by_article.transform("sum") - value) / (article_sizes - 1).replace(0, np.nan)

    neighbors = paired_neighbors(rows, article_key, scores[["qwen", "bge"]].to_numpy(dtype=np.float32))
    for column, value in zip(("qwen_nearby", "bge_nearby", "qwen_other_fragment", "bge_other_fragment"), neighbors.T):
        block[column] = value
    return pd.concat([base, scores, pd.DataFrame(block, index=rows.index)], axis=1)

class QueryRanker(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.encode = nn.Sequential(nn.Linear(size, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.08))
        self.score = nn.Sequential(nn.Linear(192, 64), nn.GELU(), nn.Dropout(0.08), nn.Linear(64, 1))
        nn.init.zeros_(self.score[-1].weight)
        nn.init.zeros_(self.score[-1].bias)

    def forward(self, features, mask, prior):
        rows = self.encode(features)
        present = mask.unsqueeze(-1)
        avg = (rows * present).sum(1, keepdim=True) / present.sum(1, keepdim=True)
        best = rows.masked_fill(~present, -torch.inf).amax(1, keepdim=True)
        group = torch.cat([rows, avg.expand_as(rows), best.expand_as(rows)], dim=-1)
        return (prior + self.score(group).squeeze(-1)).masked_fill(~mask, -10_000.)


def pack_queries(rows, values, prior):
    groups = list(rows.groupby("query_id", sort=False).indices.values())
    width = max(map(len, groups))
    index = np.full((len(groups), width), -1, dtype=np.int64)
    for offset, group in enumerate(groups):
        index[offset, :len(group)] = group
    mask = index >= 0
    safe = index.clip(min=0)
    arrays = [torch.from_numpy(values[safe]), torch.from_numpy(mask),
              torch.from_numpy(np.asarray(prior, dtype=np.float32)[safe])]
    if "relevance" in rows and rows["relevance"].notna().all():
        arrays.append(torch.from_numpy(rows["relevance"].to_numpy(dtype=np.float32)[safe] * mask))
    return index, arrays


def numeric_inputs(fit_features, next_features):
    names = fit_features.select_dtypes(include="number").columns
    left = fit_features[names].to_numpy(dtype=np.float32, copy=True)
    right = next_features[names].to_numpy(dtype=np.float32, copy=True)
    left[~np.isfinite(left)] = np.nan
    right[~np.isfinite(right)] = np.nan
    mid = np.nan_to_num(np.nanmedian(left, axis=0))
    spread = np.nan_to_num(np.nanstd(left, axis=0))
    spread[spread < 1e-5] = 1
    return (np.nan_to_num(((left - mid) / spread).clip(-8, 8)),
            np.nan_to_num(((right - mid) / spread).clip(-8, 8)))


def lambda_loss(logits, labels, valid):
    width = logits.shape[1]
    discount = 1 / torch.log2(torch.arange(width, dtype=logits.dtype) + 2)
    discount[10:] = 0
    idcg = torch.cat([discount.new_zeros(1), discount.cumsum(0)])[labels.sum(1).long()]
    order = logits.detach().argsort(dim=1, descending=True)
    rank = torch.empty_like(order).scatter_(1, order, torch.arange(width).expand_as(order))
    position_discount = discount[rank]
    delta = (position_discount[:, :, None] - position_discount[:, None, :]).abs()
    correct_pairs = (labels[:, :, None] > labels[:, None, :]) & valid[:, :, None] & valid[:, None, :]
    pair_weight = delta * correct_pairs / idcg.clamp_min(1e-6)[:, None, None]
    pair_cost = F.softplus(logits[:, None, :] - logits[:, :, None])
    lambda_rank = (pair_cost * pair_weight).sum((1, 2)) / correct_pairs.sum((1, 2)).clamp_min(1)
    label_distribution = labels / labels.sum(1, keepdim=True).clamp_min(1)
    list_net = -(label_distribution * F.log_softmax(logits, dim=1)).sum(1)
    return (lambda_rank + 0.2 * list_net).mean()


@torch.inference_mode()
def predict_query_ranker(model, packed):
    index, arrays = packed
    model.eval()
    logits = torch.cat([model(*batch) for batch in zip(*(a.split(128) for a in arrays[:3]))]).numpy()
    mask = index >= 0
    result = np.empty(int(mask.sum()), dtype=np.float32)
    result[index[mask]] = logits[mask]
    return result


def train_query_ranker(fit_rows, fit_features, next_rows, next_features, epochs):
    left, right = numeric_inputs(fit_features, next_features)
    packed_fit = pack_queries(fit_rows, left, fit_features["semantic_consensus"])
    packed_next = pack_queries(next_rows, right, next_features["semantic_consensus"])
    _, arrays = packed_fit
    torch.manual_seed(SEED)
    model = QueryRanker(left.shape[1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0012, weight_decay=0.02)
    for epoch in range(1, epochs + 1):
        model.train()
        for indexes in torch.randperm(len(arrays[0])).split(48):
            features, mask, prior, labels = (array[indexes] for array in arrays)
            optimizer.zero_grad(set_to_none=True)
            loss = lambda_loss(model(features, mask, prior), labels, mask)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            optimizer.step()
    return predict_query_ranker(model, packed_next)

def list_loss_gpu(logits, labels, mask):
    length = logits.shape[1]
    positions = torch.arange(length, device=logits.device)
    discount = 1 / torch.log2(positions + 2.)
    discount[10:] = 0
    ideal = torch.cat([discount.new_zeros(1), discount.cumsum(0)])[labels.sum(1).long()]
    order = logits.detach().argsort(dim=1, descending=True)
    rank = torch.empty_like(order).scatter_(1, order, positions.expand_as(order))
    assigned = discount[rank]
    delta = (assigned[:, :, None] - assigned[:, None, :]).abs()
    positive_pairs = (labels[:, :, None] > labels[:, None, :]) & mask[:, :, None] & mask[:, None, :]
    weights = delta * positive_pairs / ideal.clamp_min(1e-6)[:, None, None]
    pair = F.softplus(logits[:, None, :] - logits[:, :, None])
    per_query = (weights * pair).sum((1, 2)) / positive_pairs.sum((1, 2)).clamp_min(1)
    pointwise = F.binary_cross_entropy_with_logits(logits[mask], labels[mask])
    return per_query.mean() + 0.12 * pointwise


def batch_query_positions(rows, tokenized, rng):
    groups = list(rows.groupby("query_id", sort=False).indices.values())
    key = [max(len(tokenized[i]["input_ids"]) for i in group) // 48 for group in groups]
    # Shuffle groups within length buckets, then shuffle the resulting batches.
    orders = rng.permutation(len(groups)).tolist()
    orders.sort(key=key.__getitem__)
    batches, batch, count = [], [], 0
    for offset in orders:
        group = groups[offset]
        if count + len(group) > 44 and len(batch):
            batches.append(batch)
            batch, count = [], 0
        batch.append(group)
        count += len(group)
    if batch:
        batches.append(batch)
    rng.shuffle(batches)
    return batches


def setup_bge_for_training(device):
    tokenizer = AutoTokenizer.from_pretrained(BGE, revision=BGE_REV, cache_dir=MODEL_CACHE, trust_remote_code=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        BGE, revision=BGE_REV, cache_dir=MODEL_CACHE, trust_remote_code=False,
        use_safetensors=True, dtype=torch.float32, device_map={"": device}, attn_implementation="sdpa",
    )
    frozen = [model.base_model.embeddings, *model.base_model.encoder.layer[:-4]]
    for layer in frozen:
        layer.requires_grad_(False)
    return tokenizer, model, frozen


def train_bge(rows, test, *, gpu, epochs, on_epoch=None):
    device = torch.device(f"cuda:{gpu}")
    rng = np.random.default_rng(SEED + gpu)
    with torch.cuda.device(device):
        torch.cuda.manual_seed(SEED + gpu)
    tokenizer, model, frozen = setup_bge_for_training(device)
    encoded = bert_input(tokenizer, rows)
    val_encoded = bert_input(tokenizer, test)
    targets = rows["relevance"].to_numpy(dtype=np.float32)
    parameters = [value for value in model.parameters() if value.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr=1.25e-5, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda")
    for epoch in range(1, epochs + 1):
        started = perf_counter()
        model.train()
        for layer in frozen:
            layer.eval()
        batches = batch_query_positions(rows, encoded, rng)
        for step, groups in enumerate(batches, start=1):
            all_rows = np.concatenate(groups)
            width = max(map(len, groups))
            indexes = np.full((len(groups), width), -1, dtype=np.int64)
            cursor = 0
            for j, group in enumerate(groups):
                indexes[j, :len(group)] = np.arange(cursor, cursor + len(group))
                cursor += len(group)
            inside = torch.from_numpy(indexes).to(device)
            mask = inside >= 0
            indexes_clamped = indexes.clip(min=0)
            labels = torch.from_numpy(targets[all_rows[indexes_clamped]]).to(device) * mask
            batch = tokenizer.pad([encoded[i] for i in all_rows], pad_to_multiple_of=8, return_tensors="pt").to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                scores = model(**batch).logits[:, 0].float()
            logits = scores[inside.clamp_min(0)].masked_fill(~mask, -10_000.)
            scaler.scale(list_loss_gpu(logits, labels, mask)).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(parameters, 1.)
            scaler.step(optimizer)
            scaler.update()
            if step % 100 == 0:
                print(f"TRAIN BGE/{gpu} epoch {epoch}: {step}/{len(batches)} | {perf_counter()-started:.1f}s", flush=True)
        model.eval()
        if on_epoch is not None or epoch == epochs:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                predictions = score_batches(tokenizer, model, val_encoded, device, f"BGE trained/{gpu}")
            if on_epoch is not None:
                on_epoch(epoch, predictions)
        print(f"BGE/{gpu} epoch {epoch} complete | {perf_counter()-started:.1f}s", flush=True)
    return predictions

def full_bge_predictions(train, test, epochs):
    def one_seed(gpu):
        predictions = []

        def collect(_, scores):
            predictions.append(scores)

        train_bge(train, test, gpu=gpu, epochs=epochs, on_epoch=collect)
        return predictions

    with ThreadPoolExecutor(max_workers=2) as pool:
        first = pool.submit(one_seed, 0)
        second = pool.submit(one_seed, 1)
        outputs = list(zip(first.result(), second.result()))
    groups = test["query_id"]
    result = pd.DataFrame({"id": test["id"]})
    for epoch, (left, right) in enumerate(outputs, 1):
        result[f"epoch_{epoch}"] = 0.5 * (standardized(left, groups) + standardized(right, groups))
    return result

def previous_predictions(train, test, semantic):
    fit_state = fit_feature_state(train)
    train_semantic = semantic.iloc[:len(train)].set_axis(train.index)
    test_semantic = semantic.iloc[len(train):].set_axis(test.index)
    train_features = context_features(train, make_features(train, fit_state), train_semantic)
    test_features = context_features(test, make_features(test, fit_state), test_semantic)
    listwise = train_query_ranker(train, train_features, test, test_features, epochs=7)
    tuned = full_bge_predictions(train, test, 2)["epoch_2"]
    groups = test["query_id"]
    return 0.85 * standardize_scores(listwise, groups) + 0.15 * standardize_scores(tuned, groups)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter
import gc

import torch
from torch.nn import functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer


WINDOW_REPO = "BAAI/bge-reranker-v2-m3"
WINDOW_REVISION = "953dc6f"
WINDOW_EPOCHS = 2


def focus_span(query, document):
    words = list(re.finditer(r"\S+", document))
    window_size = 100
    if len(words) <= window_size + 12:
        return document
    terms = analyze_text(query)
    weighted = {token: max(1.0, len(token) - 2.0) for token in terms.stems if len(token) > 2}
    if not weighted:
        return document
    starts = list(range(0, len(words) - window_size, 30)) + [len(words) - window_size]
    spans = [document[words[start].start():words[start + window_size - 1].end()] for start in starts]

    def match(span):
        evidence = analyze_text(span)
        return sum(weighted[token] for token in weighted.keys() & evidence.counts.keys()) + 4 * len(terms.numbers & evidence.numbers)

    strongest = max(spans, key=match)
    return strongest if match(strongest) > 0 else document


def window_encodings(data, tokenizer):
    rows = data.rows
    query = rows["query"].tolist()
    text = rows["text"].tolist()
    title = rows["title"].tolist()
    crop = [focus_span(q, doc) if len(doc) > 770 else doc for q, doc in zip(query, text)]
    updated = np.array([original != focused for original, focused in zip(text, crop)])

    def encode(indices, documents):
        prompts = [f"{title[i]}\n{documents[i]}".strip() for i in indices]
        payload = tokenizer([query[i] for i in indices], prompts, truncation=True, max_length=384)
        return [dict(zip(payload, values)) for values in zip(*payload.values())]

    complete = encode(range(len(rows)), text)
    alt_index = np.flatnonzero(updated)
    alternative = dict(zip(alt_index, encode(alt_index, crop)))
    print(f"WINDOW VIEWS {len(complete):,} pairs, {len(alternative):,} with focused spans", flush=True)
    return complete, alternative


def window_model(device):
    model = AutoModelForSequenceClassification.from_pretrained(
        WINDOW_REPO, revision=WINDOW_REVISION, cache_dir=MODEL_CACHE, use_safetensors=True,
        trust_remote_code=False, dtype=torch.float32, device_map={"": device}, attn_implementation="sdpa",
    )
    frozen = [model.base_model.embeddings, *model.base_model.encoder.layer[:-8]]
    for layer in frozen:
        layer.requires_grad_(False)
    return model, frozen


def span_predictions(model, tokenizer, complete, alternative, rows, device):
    model.eval()
    indices = rows.index.to_numpy()
    full_tokens = [complete[index] for index in indices]
    changed = np.array([i for i, index in enumerate(indices) if index in alternative], dtype=int)
    cropped_tokens = [alternative[indices[i]] for i in changed]
    with torch.amp.autocast("cuda", dtype=torch.float16):
        full = score_batches(tokenizer, model, full_tokens, device, "BGE/full view", batch_size=72)
        focused = full.copy()
        if len(changed):
            focused[changed] = score_batches(tokenizer, model, cropped_tokens, device, "BGE/focused view", batch_size=72)
    groups = rows["query_id"]
    return 0.72 * standardize_scores(full, groups).to_numpy() + 0.28 * standardize_scores(focused, groups).to_numpy()


def fit_answer_windows(data, tokenizer, views, fit, holdout, *, gpu, tag, deadline, epochs=WINDOW_EPOCHS):
    device = torch.device(f"cuda:{gpu}")
    torch.cuda.manual_seed(SEED + gpu)
    rng = np.random.default_rng(SEED + gpu)
    complete, alternative = views
    train_groups = candidate_groups(fit)
    model, frozen = window_model(device)
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr=1.1e-5, weight_decay=0.01)
    scale = torch.amp.GradScaler("cuda")
    predictions = {}
    for epoch in range(1, epochs + 1):
        started = perf_counter()
        model.train()
        for layer in frozen:
            layer.eval()
        encoded = complete.copy()
        for index, focused in alternative.items():
            if rng.random() < 0.40:
                encoded[index] = focused
        lengths = np.array([len(row["input_ids"]) for row in encoded])
        batches = batch_groups(train_groups, lengths, 40, 13824, rng)
        for step, groups in enumerate(batches, 1):
            if step % 32 == 1 and perf_counter() >= deadline:
                raise RuntimeError("Answer-window model reached its GPU budget")
            optimizer.param_groups[0]["lr"] = 1.1e-5 * (min(1.0, step / 28) if epoch == 1 else 0.8)
            _, tokens, gather, valid, labels, *_ = pack_candidates(tokenizer, encoded, data, groups, device, context=False)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                pair = model(**tokens).logits[:, 0].float()
            logits = pair[gather].masked_fill(~valid, -10_000.0)
            loss = ranking_loss(logits, labels, valid)
            scale.scale(loss).backward()
            scale.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(parameters, 1.0)
            scale.step(optimizer)
            scale.update()
            if step == 1 or step % 180 == 0:
                print(f"WINDOW TRAIN {tag} e{epoch} {step}/{len(batches)}", flush=True)
        predictions[epoch] = span_predictions(model, tokenizer, complete, alternative, holdout, device)
        if not np.isfinite(predictions[epoch]).all():
            raise ValueError(f"Non-finite evidence-window predictions for {tag}")
        print(f"WINDOW EPOCH {tag} e{epoch} {(perf_counter()-started)/60:.1f}m", flush=True)
    return predictions


def train_windows(data, jobs, gpu, deadline, epochs=WINDOW_EPOCHS):
    with torch.cuda.device(gpu):
        tokenizer = AutoTokenizer.from_pretrained(WINDOW_REPO, revision=WINDOW_REVISION, cache_dir=MODEL_CACHE,
                                                  trust_remote_code=False)
        views = window_encodings(data, tokenizer)
        predictions = {}
        for tag, fit, holdout in jobs:
            predictions[tag] = fit_answer_windows(data, tokenizer, views, fit, holdout, gpu=gpu, tag=tag,
                                                  deadline=deadline, epochs=epochs)
            gc.collect()
            torch.cuda.empty_cache()
    return predictions


def result_weights(rows, weight, mode):
    if mode == "fixed":
        return weight
    long_fragments = rows["text"].str.len().gt(900)
    density = long_fragments.groupby(rows["query_id"], sort=False).transform("mean")
    return weight * (0.30 + 0.70 * density.to_numpy())

In [ ]:
def final_test_predictions(data, fit, test, test_reference, selection, deadline):
    total = selection["reference_weight"] * standardize_scores(test_reference, test["query_id"]).to_numpy()
    name = "joint_bge"
    weight = selection[f"{name}_weight"]
    if not weight:
        return total
    epoch = int(selection[f"{name}_epoch"])
    jobs = [("full", fit, test, epoch, False)]
    with ThreadPoolExecutor(max_workers=2) as executor:
        results = [executor.submit(run_encoder, name, data, jobs, gpu, deadline, seed_offset=gpu) for gpu in range(2)]
        logits = [result.result()["full"][epoch][test.index] for result in results]
    mean = sum(standardize_scores(scores, test["query_id"]).to_numpy() for scores in logits) / len(logits)
    return total + weight * mean


started = perf_counter()
deadline = started + 10.75 * 3600
configure_accelerator()
torch.manual_seed(SEED)
print(f"{platform.system()} | Python {platform.python_version()}", flush=True)
packages = ("accelerate", "huggingface-hub", "nltk", "numpy", "pandas", "pyarrow",
            "safetensors", "tokenizers", "torch", "transformers")
print("PACKAGE VERSIONS " + json.dumps({package: version(package) for package in packages}, sort_keys=True), flush=True)
raw_train, raw_test, sample = load_data()
validate_template(raw_test, sample)
train, test = prepare_pairs(raw_train), prepare_pairs(raw_test)
pairs = pd.concat([train, test], ignore_index=True)
semantic = encode_semantics(pairs, Path("/kaggle/temp/amml_v3_semantics.parquet"))
if tuple(semantic.columns) != SEMANTIC_COLUMNS or not np.isfinite(semantic.to_numpy()).all():
    raise ValueError("Invalid semantic score columns or values")
reference = previous_predictions(train, test, semantic)
data = CandidateData(pairs, semantic)
test = pairs.iloc[len(train):]
selection = {'reference_weight': 0.7, 'joint_bge_weight': 0.3, 'joint_bge_epoch': 2, 'tuned_qwen_weight': 0, 'tuned_qwen_epoch': 0}
predictions = final_test_predictions(data, train, test, reference, selection, deadline)
jobs = [("full", train, test)]
window_epoch = 1
with ThreadPoolExecutor(max_workers=2) as workers:
    left = workers.submit(train_windows, data, jobs, 0, deadline, window_epoch)
    right = workers.submit(train_windows, data, jobs, 1, deadline, window_epoch)
    evidence = 0.5 * (left.result()["full"][window_epoch] + right.result()["full"][window_epoch])
weight = result_weights(test, 0.7, 'long')
predictions = ((1.0 - weight) * standardize_scores(predictions, test["query_id"]).to_numpy()
               + weight * standardize_scores(evidence, test["query_id"]).to_numpy())
print(f"BASE RANKING COMPLETE after {(perf_counter() - started) / 3600:.2f}h", flush=True)

from hashlib import sha256

output = Path(os.environ.get("AMML_OUTPUT_DIR", "/kaggle/working"))
output.mkdir(parents=True, exist_ok=True)
destination = output / "submission.csv"
write_submission(test, sample, predictions, destination)
saved = pd.read_csv(destination, dtype={"id": str})
if list(saved.columns) != ["id", "prediction"] or not saved["id"].equals(sample["id"]) or not np.isfinite(saved["prediction"]).all():
    raise ValueError("The saved CSV does not match the competition template")
fingerprint = sha256(destination.read_bytes()).hexdigest()
print(f"FULL RUN VERIFIED {len(saved):,} predictions after {(perf_counter() - started) / 3600:.2f}h "
      f"| SHA256 {fingerprint}", flush=True)
